# Task 4 - Game Title Detection from Screenshots (5 classes, metric: F1 macro)

CPE342 Karena Player Intelligence System. Fast version of the `main` notebook, built for Colab with the data in Google Drive.

> **Rules followed:** image classification may use pre-trained weights (ImageNet-22k ConvNeXt fine-tuned on the task images, no AutoML). Every statistic (class weights, fold split, best-epoch choice) comes from the **train** set only; the test set is used only for the final prediction.

**What changed from `main` and why**

| | `main` | this notebook | why |
|---|---|---|---|
| Image loading | PIL from disk every epoch, 2 workers | decoded **once** into a uint8 array (cached to Drive), batches sliced on the GPU | Colab has 2 CPUs; JPEG decoding was the bottleneck |
| Augmentation | CPU torchvision | per-sample random-resized-crop, flip, colour, blur, erasing **on the GPU** | no CPU work per batch |
| Input size | 224x224 (aspect squashed) | 160x320 (keeps the 230x120 aspect) | HUD/layout is not distorted |
| Backbone | ConvNeXt-Base | ConvNeXt-Tiny (in22k) | ~3x faster per image, small accuracy cost |
| Folds used | 3 of 5 | 5 of 5, each fold predicts test | full OOF + 5-model test ensemble |
| Extras | best epoch | EMA weights, warmup + cosine, AMP, `channels_last`, per-fold resume | stability and Colab disconnects |
| File names | `{id}.jpg` (files are named by `file_name`) | `file_name` column | the `main` path did not match the image files |

Expected time on a Colab **T4**: about 20-25 min for 5 folds x 6 epochs (after the one-off image cache). Set `FOLDS_TO_TRAIN = [0, 1, 2]` to cut it to ~13 min.

## 1. Environment and data (Colab: Google Drive)

In [ ]:
import os, sys, glob, time, random, shutil, subprocess, warnings, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

QUICK = os.environ.get("QUICK") == "1"   # QUICK=1 -> tiny CPU smoke test (results not meaningful)

try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])
    import timm
import torch

# ---- where the data lives -------------------------------------------------
# Colab: put train.csv, test.csv, train/ and test/ in a Drive folder (default: MyDrive/Colab Notebooks),
# or a task4*.zip anywhere in MyDrive. Set DATA_DIR by hand if it is somewhere else.
DATA_DIR = None
DRIVE_SEARCH = ["/content/drive/MyDrive/Colab Notebooks", "/content/drive/MyDrive"]
GDRIVE_FILE_ID = "1C8csylbBo3Oab24ODGKPVom3_SDUJ_Qw"   # official task4 zip (fallback download)

def looks_like_data(d):
    d = Path(d)
    return (d / "train.csv").exists() and (d / "test.csv").exists() and (d / "train").is_dir() and (d / "test").is_dir()

def find_data_dir(bases):
    for base in bases:
        for pat in ["", "*", "*/*", "*/*/*"]:
            for p in sorted(glob.glob(str(Path(base) / pat / "train.csv")) if pat else [str(Path(base) / "train.csv")]):
                if looks_like_data(Path(p).parent):
                    return Path(p).parent
    return None

if IN_COLAB:
    drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/CPE342_task4_fast")      # fold results + image cache survive a disconnect
    if DATA_DIR is None:
        DATA_DIR = find_data_dir(DRIVE_SEARCH)
    if DATA_DIR is None:                                            # no extracted folder: use a zip (copy 1 file, unzip locally)
        zips = [z for z in glob.glob("/content/drive/MyDrive/**/*.zip", recursive=True) if "task4" in Path(z).name.lower()]
        local_zip = Path("/content/task4.zip")
        if zips:
            print("copying", zips[0]); shutil.copy(zips[0], local_zip)
        else:
            print("no data in Drive, downloading the official zip")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
            subprocess.check_call(["gdown", GDRIVE_FILE_ID, "-O", str(local_zip)])
        with zipfile.ZipFile(local_zip) as z:
            z.extractall("/content/data")
        DATA_DIR = find_data_dir(["/content/data"])
else:
    ROOT = Path.cwd()
    DATA_DIR = DATA_DIR or (ROOT / "task4" if (ROOT / "task4" / "train.csv").exists() else ROOT)
    SAVE_DIR = Path(DATA_DIR) / ("outputs_quick" if QUICK else "outputs")

assert DATA_DIR is not None and looks_like_data(DATA_DIR), f"task4 data not found (DATA_DIR={DATA_DIR})"
DATA_DIR = Path(DATA_DIR)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR, "| SAVE_DIR:", SAVE_DIR, "| QUICK:", QUICK)
print("PyTorch", torch.__version__, "| timm", timm.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
elif IN_COLAB:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU")

## 2. Configuration

Change only this cell. `MODEL_NAME = "convnext_small.fb_in22k_ft_in1k"` is ~1.6x slower and usually a little more accurate.

In [ ]:
SEED = 42
TASK = 4
NAMES = ["7-11 Knights", "FiveN", "Free Fried", "PUBE", "Roblock"]
NUM_CLASSES = 5

MODEL_NAME = "convnext_tiny.fb_in22k_ft_in1k"
IMG_SIZE = (160, 320)          # (H, W); the images are 120x230, so this keeps the aspect ratio
N_FOLDS = 5
FOLDS_TO_TRAIN = [0, 1, 2, 3, 4]
EPOCHS = 6
BATCH = 128
LR = 3e-4                      # backbone; the head gets LR * HEAD_LR_MULT
HEAD_LR_MULT = 5.0
WEIGHT_DECAY = 0.05
WARMUP_FRAC = 0.05
EMA_DECAY = 0.998
LABEL_SMOOTHING = 0.1
CLASS_WEIGHT_POWER = 0.5       # balanced class weights ** 0.5 (train fold only)
TTA_FLIP = True
CACHE_TO_DRIVE = True          # save the decoded uint8 images next to the results (one-off ~4.7 GB, makes re-runs fast)

if QUICK:
    MODEL_NAME, IMG_SIZE, N_FOLDS, FOLDS_TO_TRAIN, EPOCHS, BATCH = "convnext_atto.d2_in1k", (64, 128), 2, [0, 1], 1, 32
    CACHE_TO_DRIVE = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_CUDA = DEVICE == "cuda"
# bf16 only on Ampere or newer (A100/L4); T4 (sm75) reports bf16 "supported" but cuDNN has no bf16 conv kernels for it
AMP_DTYPE = torch.bfloat16 if USE_CUDA and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
RUN_TAG = f"{MODEL_NAME.split('.')[0]}_{IMG_SIZE[0]}x{IMG_SIZE[1]}_e{EPOCHS}"

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(f"{RUN_TAG} | device {DEVICE} | AMP {AMP_DTYPE if USE_CUDA else 'off'} | folds {FOLDS_TO_TRAIN}")

## 3. Load the tables and decode every image once

Reading ~57k small JPEGs from Drive is the slowest step, so it is done once with 32 threads and cached as one `.npy` file per split.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
TARGET = "label" if "label" in train.columns else "class"
FILE_COL = "file_name" if "file_name" in train.columns else "id"
if QUICK:
    train = train.sample(frac=1, random_state=SEED).groupby(TARGET).head(60).reset_index(drop=True)
    test = test.head(200).reset_index(drop=True)
print(f"train {train.shape} | test {test.shape} | target '{TARGET}' | files '{FILE_COL}'")

from PIL import Image
NATIVE = (120, 230)   # (H, W) from the datacard

def path_of(split, name):
    name = str(name)
    return DATA_DIR / split / (name if name.lower().endswith(".jpg") else name + ".jpg")

def decode(p):
    im = Image.open(p).convert("RGB")
    if im.size != (NATIVE[1], NATIVE[0]):
        im = im.resize((NATIVE[1], NATIVE[0]), Image.BILINEAR)
    return np.asarray(im, dtype=np.uint8)

def load_split(split, df):
    cache = SAVE_DIR / f"cache_{split}_{len(df)}.npy"
    local = Path("/content") / cache.name if IN_COLAB else cache
    for c in (local, cache):
        if CACHE_TO_DRIVE or c == local:
            if c.exists():
                print(f"{split}: loading cache {c}")
                return np.load(c)
    t0 = time.time()
    arr = np.empty((len(df), *NATIVE, 3), dtype=np.uint8)
    paths = [path_of(split, n) for n in df[FILE_COL]]
    def work(i):
        arr[i] = decode(paths[i])
    with ThreadPoolExecutor(32) as ex:
        for k, _ in enumerate(ex.map(work, range(len(paths)))):
            if (k + 1) % 5000 == 0:
                print(f"  {split}: {k + 1}/{len(paths)} ({time.time() - t0:.0f}s)")
    print(f"{split}: decoded {len(df)} images in {time.time() - t0:.0f}s ({arr.nbytes / 1e9:.2f} GB)")
    if IN_COLAB:
        np.save(local, arr)
    if CACHE_TO_DRIVE:
        np.save(cache, arr)
    return arr

X_train = load_split("train", train)
X_test = load_split("test", test)
y_all = train[TARGET].astype(int).values

## 4. EDA (train only)

In [ ]:
import matplotlib.pyplot as plt
vc = pd.Series(y_all).value_counts(normalize=True).sort_index().set_axis(NAMES)
print(vc.round(3))
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(15, 8))
rng = np.random.RandomState(SEED)
for k, row in enumerate(axes):
    idx = np.where(y_all == k)[0]
    for ax, i in zip(row, rng.choice(idx, min(5, len(idx)), replace=False)):
        ax.imshow(X_train[i]); ax.axis("off")
    row[0].set_title(NAMES[k], loc="left", fontsize=10)
plt.tight_layout(); plt.show()

## 5. GPU batches and augmentation

Each batch is sliced from the uint8 array, moved to the GPU, then augmented per sample:
random-resized-crop (60-100 % area, aspect 3/4-4/3) and horizontal flip in one `affine_grid` call that also resizes to `IMG_SIZE`;
brightness / contrast / saturation jitter; grayscale (10 %); light blur (15 %); random erasing (25 %).
Test screenshots come from other sources than train, so the model should not depend on exact framing or colour.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

# keep train images on the GPU when there is room (T4 16 GB: yes), otherwise slice on the CPU
GPU_RESIDENT = USE_CUDA and torch.cuda.get_device_properties(0).total_memory > 14e9
X_train_src = torch.from_numpy(X_train).to(DEVICE) if GPU_RESIDENT else torch.from_numpy(X_train)
print("train images resident on GPU:", GPU_RESIDENT)

def to_float(xb_u8):
    x = xb_u8.to(DEVICE, non_blocking=True).permute(0, 3, 1, 2).float().div_(255)
    return x

def resize_eval(x):
    return F.interpolate(x, size=IMG_SIZE, mode="bilinear", align_corners=False, antialias=True)

def augment(x):
    B, dev = x.shape[0], x.device
    u = lambda lo, hi: torch.empty(B, device=dev).uniform_(lo, hi)
    area, logr = u(0.6, 1.0), u(np.log(3 / 4), np.log(4 / 3))
    sx = torch.sqrt(area * torch.exp(logr)).clamp(max=1.0)
    sy = torch.sqrt(area / torch.exp(logr)).clamp(max=1.0)
    tx, ty = (1 - sx) * u(-1, 1), (1 - sy) * u(-1, 1)
    flip = torch.where(torch.rand(B, device=dev) < 0.5, -1.0, 1.0)
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 0, 2], theta[:, 1, 1], theta[:, 1, 2] = sx * flip, tx, sy, ty
    grid = F.affine_grid(theta, (B, 3, *IMG_SIZE), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)

    on = (torch.rand(B, 1, 1, 1, device=dev) < 0.8).float()          # colour jitter on 80 %
    b = 1 + on * (u(0.75, 1.25).view(B, 1, 1, 1) - 1)
    c = 1 + on * (u(0.75, 1.25).view(B, 1, 1, 1) - 1)
    s = 1 + on * (u(0.7, 1.3).view(B, 1, 1, 1) - 1)
    x = x * b
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    x = (x - m) * c + m
    g = (0.299 * x[:, :1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3])
    x = g + (x - g) * s
    gray = (torch.rand(B, 1, 1, 1, device=dev) < 0.10).float()
    x = x * (1 - gray) + g.expand_as(x) * gray
    blur = (torch.rand(B, 1, 1, 1, device=dev) < 0.15).float()
    x = x * (1 - blur) + F.avg_pool2d(x, 3, 1, 1, count_include_pad=False) * blur
    x = x.clamp_(0, 1)
    x = (x - MEAN) / STD

    H, W = IMG_SIZE                                                    # random erasing on 25 %
    er = torch.rand(B, device=dev) < 0.25
    eh, ew = (u(0.1, 0.35) * H).long(), (u(0.1, 0.35) * W).long()
    y0 = (torch.rand(B, device=dev) * (H - eh)).long(); x0 = (torch.rand(B, device=dev) * (W - ew)).long()
    yy = torch.arange(H, device=dev).view(1, H, 1); xx = torch.arange(W, device=dev).view(1, 1, W)
    box = (yy >= y0.view(B, 1, 1)) & (yy < (y0 + eh).view(B, 1, 1)) & (xx >= x0.view(B, 1, 1)) & (xx < (x0 + ew).view(B, 1, 1))
    box = (box & er.view(B, 1, 1)).unsqueeze(1)
    return torch.where(box, torch.randn_like(x), x)

def prep_eval(xb_u8):
    return (resize_eval(to_float(xb_u8)) - MEAN) / STD

# visual check of the augmentation
xb = augment(to_float(X_train_src[:8]))
vis = (xb * STD + MEAN).clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()
fig, axes = plt.subplots(2, 4, figsize=(16, 4.5))
for ax, im in zip(axes.flat, vis):
    ax.imshow(im); ax.axis("off")
plt.suptitle("augmented training batch"); plt.tight_layout(); plt.show()

## 6. Training: stratified 5-fold, EMA, warmup + cosine

Per fold: AdamW (head LR x5), class-weighted cross-entropy with label smoothing, mixed precision, `channels_last`.
The EMA weights are scored on the validation fold after every epoch and the best epoch is kept.
Every finished fold is saved to `SAVE_DIR`, so a re-run after a Colab disconnect continues where it stopped.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from timm.utils import ModelEmaV2

folds = np.full(len(train), -1)
for f, (_, va) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y_all)), y_all)):
    folds[va] = f

def autocast():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if USE_CUDA else torch.autocast("cpu", enabled=False)

@torch.no_grad()
def predict(model, src, idx=None, tta=TTA_FLIP, bs=None):
    model.eval()
    bs = bs or BATCH * 2
    n = len(idx) if idx is not None else len(src)
    out = []
    for s in range(0, n, bs):
        sel = idx[s:s + bs] if idx is not None else slice(s, s + bs)
        xb = src[torch.as_tensor(sel, device=src.device) if idx is not None else sel]
        x = prep_eval(xb).contiguous(memory_format=torch.channels_last)
        with autocast():
            logit = model(x).float()
            p = logit.softmax(1)
            if tta:
                p = (p + model(torch.flip(x, dims=[3])).float().softmax(1)) / 2
        out.append(p.cpu().numpy())
    return np.concatenate(out)

def run_fold(fold):
    cache = SAVE_DIR / f"{RUN_TAG}_fold{fold}.npz"
    if cache.exists():
        d = np.load(cache)
        print(f"fold {fold}: loaded from {cache.name} (macro-F1 {float(d['f1']):.5f})")
        return d["val_idx"], d["val_p"], d["test_p"], float(d["f1"])

    seed_everything(SEED + fold)
    tr_idx, va_idx = np.where(folds != fold)[0], np.where(folds == fold)[0]
    cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_all[tr_idx])
    crit = nn.CrossEntropyLoss(weight=torch.tensor(cw ** CLASS_WEIGHT_POWER, dtype=torch.float, device=DEVICE),
                               label_smoothing=LABEL_SMOOTHING)
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES, drop_path_rate=0.1)
    model = model.to(DEVICE, memory_format=torch.channels_last)
    head_ids = {id(p) for p in model.get_classifier().parameters()}
    params = [{"params": [p for p in model.parameters() if id(p) not in head_ids], "lr": LR},
              {"params": [p for p in model.parameters() if id(p) in head_ids], "lr": LR * HEAD_LR_MULT}]
    opt = torch.optim.AdamW(params, weight_decay=WEIGHT_DECAY, **({"fused": True} if USE_CUDA else {}))
    steps_per_ep = len(tr_idx) // BATCH
    total = EPOCHS * steps_per_ep
    warm = max(1, int(WARMUP_FRAC * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + np.cos(np.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_CUDA and AMP_DTYPE == torch.float16)
    ema = ModelEmaV2(model, decay=EMA_DECAY)
    y_dev = torch.as_tensor(y_all, device=DEVICE)

    best_f1, best_state, best_val = -1.0, None, None
    for ep in range(EPOCHS):
        model.train(); t0 = time.time(); loss_sum = 0.0
        perm = np.random.permutation(tr_idx)
        for s in range(steps_per_ep):
            sel = perm[s * BATCH:(s + 1) * BATCH]
            xb = X_train_src[torch.as_tensor(sel, device=X_train_src.device)]
            x = augment(to_float(xb)).contiguous(memory_format=torch.channels_last)
            yb = y_dev[torch.as_tensor(sel, device=DEVICE)]
            with autocast():
                loss = crit(model(x), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt); scaler.update(); sched.step()
            ema.update(model)
            loss_sum += loss.item()
        pv = predict(ema.module, X_train_src, va_idx, tta=False)
        f1 = f1_score(y_all[va_idx], pv.argmax(1), average="macro")
        print(f"fold {fold} | epoch {ep + 1}/{EPOCHS} | loss {loss_sum / steps_per_ep:.4f} | EMA val macro-F1 {f1:.5f} | {time.time() - t0:.0f}s")
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().clone() for k, v in ema.module.state_dict().items()}
    ema.module.load_state_dict(best_state)
    val_p = predict(ema.module, X_train_src, va_idx, tta=TTA_FLIP)
    X_test_src = torch.from_numpy(X_test)
    test_p = predict(ema.module, X_test_src)
    f1_tta = f1_score(y_all[va_idx], val_p.argmax(1), average="macro")
    print(f"fold {fold} best EMA macro-F1 {best_f1:.5f} | with flip-TTA {f1_tta:.5f}")
    np.savez(cache, val_idx=va_idx, val_p=val_p, test_p=test_p, f1=f1_tta)
    torch.save(best_state, SAVE_DIR / f"{RUN_TAG}_fold{fold}_ema.pt")
    del model, ema, opt, best_state
    if USE_CUDA:
        torch.cuda.empty_cache()
    return va_idx, val_p, test_p, f1_tta

In [ ]:
oof = np.full((len(train), NUM_CLASSES), np.nan)
test_p = np.zeros((len(test), NUM_CLASSES))
fold_f1 = {}
T0 = time.time()
for fold in FOLDS_TO_TRAIN:
    vi, pv, pt, f1 = run_fold(fold)
    oof[vi] = pv
    test_p += pt / len(FOLDS_TO_TRAIN)
    fold_f1[fold] = f1
    print(f"--- elapsed {(time.time() - T0) / 60:.1f} min")

done = ~np.isnan(oof[:, 0])
print("\nper-fold macro-F1:", {k: round(v, 5) for k, v in fold_f1.items()})
print(f"OOF macro-F1 on {done.sum()} rows ({len(FOLDS_TO_TRAIN)}/{N_FOLDS} folds): "
      f"{f1_score(y_all[done], oof[done].argmax(1), average='macro'):.5f}")
print(classification_report(y_all[done], oof[done].argmax(1), labels=list(range(NUM_CLASSES)), target_names=NAMES, digits=4, zero_division=0))
cm = confusion_matrix(y_all[done], oof[done].argmax(1), labels=list(range(NUM_CLASSES)), normalize="true")
plt.figure(figsize=(6, 5)); plt.imshow(cm, cmap="Blues", vmin=0, vmax=1); plt.colorbar()
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center")
plt.xticks(range(NUM_CLASSES), NAMES, rotation=30); plt.yticks(range(NUM_CLASSES), NAMES)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Confusion matrix (row-normalised, out-of-fold)")
plt.tight_layout(); plt.show()

## 7. Predict the test set and save

In [ ]:
pred = test_p.argmax(1)
share = pd.DataFrame({"train": pd.Series(y_all).value_counts(normalize=True),
                      "test predicted": pd.Series(pred).value_counts(normalize=True)}).reindex(range(NUM_CLASSES)).fillna(0).set_axis(NAMES)
print(share.round(3))

df_pred = pd.DataFrame({"id": test["id"].values, "pred": pred})
for k in range(NUM_CLASSES):
    df_pred[f"p{k}"] = test_p[:, k]
df_pred.to_csv(SAVE_DIR / "task4_pred.csv", index=False)
pd.DataFrame({"id": test["id"].values, "task4": pred}).to_csv(SAVE_DIR / "task4_only_submission.csv", index=False)
np.save(SAVE_DIR / f"{RUN_TAG}_oof_proba.npy", oof)
np.save(SAVE_DIR / f"{RUN_TAG}_test_proba.npy", test_p)
print("saved to", SAVE_DIR, ":", ["task4_pred.csv", "task4_only_submission.csv", f"{RUN_TAG}_oof_proba.npy", f"{RUN_TAG}_test_proba.npy"])

# put the task4 column into a full submission if one is next to the data
for cand in [DATA_DIR / "submission_final.csv", DATA_DIR.parent / "submission_final.csv"]:
    if cand.exists() and not QUICK:
        sub = pd.read_csv(cand)
        assert (sub["id"].values == test["id"].values).all(), "id order differs from test.csv"
        sub["task4"] = pred
        sub.to_csv(SAVE_DIR / "submission_final_new_task4.csv", index=False)
        print("wrote", SAVE_DIR / "submission_final_new_task4.csv", "(task4 replaced, other columns unchanged)")
        break

if IN_COLAB:
    from google.colab import files
    files.download(str(SAVE_DIR / "task4_pred.csv"))
    files.download(str(SAVE_DIR / "task4_only_submission.csv"))